In [38]:
%pip install pytest

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
import os
from dotenv import load_dotenv
# from pymilvus import connections

# # If using Docker standalone Milvus
# connections.connect("default", host="127.0.0.1", port="19530")

from pymilvus import connections

load_dotenv(override=True, dotenv_path="../.env.local")

milvus_uri = os.getenv("MILVUS_URI")
milvus_token = os.getenv("MILVUS_API_KEY")


connections.connect(
    alias="default",
    uri=milvus_uri,
    token=milvus_token
)

print("Connected to Milvus on Zilliz Cloud")

from pymilvus import utility, Collection

# 1. List all collections to verify the name
all_collections = utility.list_collections()
print(f"Collections available: {all_collections}")

# 2. Pick your collection (ensure spelling matches the list above)
if target_col in all_collections:
    collection = Collection(target_col)
    
    # 3. CRITICAL: Check if data exists and load it
    # num_entities might show 0 if not flushed, but let's check
    print(f"Entities: {collection.num_entities}")
    
    print("Loading collection into memory...")
    collection.load() 
    print("Collection loaded and ready for search!")
else:
    print(f"Error: '{target_col}' not found. Did you mean: {all_collections[0]}?")

Connected to Milvus on Zilliz Cloud
Collections available: ['test_collection', 'policy_docs_collection', 'demo_collection']


NameError: name 'target_col' is not defined

In [19]:
from openai import OpenAI

# --- Load OpenAI API Key ---
load_dotenv(override=True, dotenv_path="../.env")
my_api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=my_api_key)


In [ ]:
#data = {
#    "question": [
#        "How often are employees paid?",
#        "What is the process for requesting leave?"
#    ],
#    "ground_truth": [
#        "Employees are paid bi-weekly via direct deposit.",
#        "Employees must submit a leave request for approval."
#    ]
#}

In [17]:
#1. Setup the Retrieval Function
# You first need a function that queries Milvus to get the content based on a question. This populates the contexts column.

from pymilvus import Collection
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Initialize the model (Make sure this matches what's in Milvus!) Common default is 'all-MiniLM-L6-v2'
embedding_model = SentenceTransformer('all-MiniLM-L6-v2') 

def get_milvus_context(query_text, collection_name):
    collection = Collection(collection_name)

    # Step 0. Check if the collection actually has data
    print(f"Collection Name: {collection.name}")
    print(f"Is Empty: {collection.is_empty}")
    print(f"Entity Count: {collection.num_entities}")

    # Step 0. Check if the collection is LOADED into memory
    # Milvus CANNOT search a collection that isn't loaded.
    from pymilvus import utility
    print(f"Load Status: {utility.load_state(collection.name)}")

    if utility.load_state(collection.name) != "Loaded":
        print("Action: Loading collection...")
        collection.load()
    
    # Step 1: Convert query text to numbers (the vector)
    # .tolist() is required so Milvus understands the data format
    query_embedding = embedding_model.encode(query_text).tolist()
    
    # Step 2: Search Milvus
    search_params = {"metric_type": "COSINE", "params": {"nprobe": 10}}
    results = collection.search(
        data=[query_embedding], 
        anns_field="embedding", 
        param=search_params,
        limit=2, #The Fix: Lower your limit=2 or shorten your chunk sizes during the embedding process.
        output_fields=["content"]
    )
    
    # Step 3: Print results so you can see them while the loop runs
    print(f"\n--- Milvus Search Results for: '{query_text}' ---")
    for i, hit in enumerate(results[0]):
        content = hit.entity.get("content")
        print(f" Match {i+1} | Score: {hit.distance:.4f} | Content: {content[:80]}...")

    # Return the list of strings for your RAG pipeline
    return [hit.entity.get("content") for hit in results[0]]

In [39]:
#2. Prepare the Evaluation Dataset
#Ragas requires a Dataset object. You can create a small "Golden Set" of questions you know the answers to (Ground Truth) and let your RAG system generate the rest.


from datasets import Dataset

# Your test questions and their known correct answers from your policy docs
data = {
    "question": [
        "How often are employees paid?",
        "What is the process for requesting leave?"
    ],
    "ground_truth": [
        "Employees are paid bi-weekly via direct deposit.",
        "Employees must submit a leave request for approval."
    ]
}

# Run your RAG pipeline to fill in 'contexts' and 'answer'
contexts = []
answers = []

for q in data["question"]:
    # 1. Retrieve from Milvus
    retrieved_docs = get_milvus_context(q, "policy_docs_collection")
    contexts.append(retrieved_docs)
    
    # 2. CREATE THE PROMPT (This is the instruction for the AI)
    # We tell it to be SUCCINCT so Ragas doesn't crash later
    prompt = f"Answer the question using ONLY the provided context. Be succinct (1-2 sentences).\n\nContext: {retrieved_docs}\n\nQuestion: {q}"
    
    # 3. CALL OPENAI (This defines the 'response')
    completion = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}]
    )
    
    response = completion.choices[0].message.content
    
    # 4. Save the answer
    answers.append(response)

data["retrieved_contexts"] = contexts
data["answer"] = answers

dataset = Dataset.from_dict(data)
print("Dataset ready with short answers!")

Collection Name: policy_docs_collection
Is Empty: False
Entity Count: 11
Load Status: Loaded
Action: Loading collection...

--- Milvus Search Results for: 'How often are employees paid?' ---
 Match 1 | Score: 0.6337 | Content: Employees are paid bi-weekly via direct deposit....
 Match 2 | Score: 0.4749 | Content: Employees can take an hour break....
Collection Name: policy_docs_collection
Is Empty: False
Entity Count: 11
Load Status: Loaded
Action: Loading collection...

--- Milvus Search Results for: 'What is the process for requesting leave?' ---
 Match 1 | Score: 0.7378 | Content: Employees must submit a leave request for approval....
 Match 2 | Score: 0.2366 | Content: Employees can take an hour break....
Dataset ready with short answers!


In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
#from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# What to look for in your scores:
# Faithfulness < 0.8: Your LLM is likely bringing in outside knowledge. Try making your system prompt stricter.
# Context Precision < 0.8: Milvus found something, but it wasn't the most relevant chunk. You might need to increase your limit (top_k) or improve your chunking.
# Answer Relevancy < 0.8: The answer is technically correct, but it contains a lot of "fluff," preamble, or redundant information that wasn't asked for.

# --- STEP 3: RUN EVALUATION ---
# We map your data keys to what Ragas expects
ragas_input_data = {
    "question": data["question"],          
    "contexts": data["retrieved_contexts"], # The list of lists from Milvus
    "answer": data["answer"],               
    "ground_truth": data["ground_truth"]    
}

eval_dataset = Dataset.from_dict(ragas_input_data)

# Define the LLM (The Judge)
#evaluator_llm = ChatOpenAI(model="gpt-4o-mini", max_tokens=1000)
# Define the Embeddings (The Vectorizer)
# This fixes the 'AttributeError' by giving Ragas the correct tool
#evaluator_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

result = evaluate(
    dataset=eval_dataset,
    metrics=[
        faithfulness,       # Did the AI stick to the docs?
        answer_relevancy,   # Is the answer actually helpful?
        context_precision   # Did Milvus find the best chunk?
    ]
    #,llm=evaluator_llm # Tell Ragas to use this specific config
    #,embeddings=evaluator_embeddings
)

# --- STEP 4: VIEW THE RESULTS ---
print("\n--- RAGAS SCORECARD ---")
print(result)

# Convert to a dataframe to see which specific questions did best
df = result.to_pandas()
# Check what columns actually exist (optional debugging)
print(df.columns) 

# Use the names Ragas generated:
# 'user_input' is the new name for 'question'
# 'response' is the new name for 'answer'
display(df[['user_input', 'retrieved_contexts', 'faithfulness', 'answer_relevancy']])


C:\Users\nirma\AppData\Local\Temp\ipykernel_47924\532526799.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
C:\Users\nirma\AppData\Local\Temp\ipykernel_47924\532526799.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
C:\Users\nirma\AppData\Local\Temp\ipykernel_47924\532526799.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import c

Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[1]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[4]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')



--- RAGAS SCORECARD ---
{'faithfulness': 1.0000, 'answer_relevancy': nan, 'context_precision': 1.0000}
Index(['user_input', 'retrieved_contexts', 'response', 'reference',
       'faithfulness', 'answer_relevancy', 'context_precision'],
      dtype='object')


,user_input,retrieved_contexts,faithfulness,answer_relevancy
0,How often are employees paid?,[Employees are paid bi-weekly via direct depos...,1.0,NaN
1,What is the process for requesting leave?,[Employees must submit a leave request for app...,1.0,NaN


In [43]:
# Export the evaluation results to csv
import pandas as pd

# 1. Convert the Ragas result object to a Pandas DataFrame
df_results = result.to_pandas()

# 2. Define your filename
csv_filename = "ragas_evaluation_report.csv"

# 3. Export to CSV
# index=False prevents an extra 'index' column in your Excel/Sheets
df_results.to_csv(csv_filename, index=False)

print(f"Success! Your scorecard is saved as: {csv_filename}")

# Optional: Print the first few rows to verify
print(df[['user_input', 'retrieved_contexts', 'faithfulness', 'answer_relevancy']])


Success! Your scorecard is saved as: ragas_evaluation_report.csv
                                  user_input  \
0              How often are employees paid?   
1  What is the process for requesting leave?   

                                  retrieved_contexts  faithfulness  \
0  [Employees are paid bi-weekly via direct depos...           1.0   
1  [Employees must submit a leave request for app...           1.0   

   answer_relevancy  
0               NaN  
1               NaN  


In [13]:
import json
import random
from pymilvus import connections, Collection
import os
from dotenv import load_dotenv

# # If using Docker standalone Milvus
# connections.connect("default", host="127.0.0.1", port="19530")

def create_finetuning_data(collection_name, output_prefix):
    connections.connect(
    alias="default",
    uri=milvus_uri,
    token=milvus_token
    )
    # 1. Fetch all data from Milvus
    # We query for all entities (limit 1000 or your total count)
    collection = Collection(collection_name)
    res = collection.query(expr="doc_id >= 0", output_fields=["content"], limit=1000)
    
    # 2. Format into OpenAI 'messages' structure
    dataset = []
    for item in res:
        content = item["content"]
        # We create a 'synthetic' pair: a general question about this chunk
        # In a real scenario, you'd use your 'Golden Set' of Q&A
        entry = {
            "messages": [
                {"role": "system", "content": "You are a helpful HR assistant for our company."},
                {"role": "user", "content": f"Summarize this policy: {content[:100]}..."},
                {"role": "assistant", "content": content}
            ]
        }
        dataset.append(entry)

    # 3. Split into Train (80%) and Validation (20%)
    random.shuffle(dataset)
    split_index = int(len(dataset) * 0.8)
    train_data = dataset[:split_index]
    val_data = dataset[split_index:]

    # 4. Save to JSONL
    for name, data in [("train", train_data), ("val", val_data)]:
        filename = f"{output_prefix}_{name}.jsonl"
        with open(filename, "w") as f:
            for entry in data:
                f.write(json.dumps(entry) + "\n")
        print(f"Saved {len(data)} rows to {filename}")

# Run it
create_finetuning_data("policy_docs_collection", "policy_ft")


Saved 7 rows to policy_ft_train.jsonl
Saved 2 rows to policy_ft_val.jsonl


In [36]:
import json

def check_jsonl(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        print(f"Checking {filename}: {len(lines)} lines found.")
        if len(lines) < 10:
            print("ERROR: OpenAI requires at least 10 examples.")
        for i, line in enumerate(lines):
            try:
                data = json.loads(line)
                # Check for required chat keys
                if "messages" not in data:
                    print(f"Line {i}: Missing 'messages' key")
            except Exception as e:
                print(f"Line {i}: Invalid JSON - {e}")

check_jsonl("policy_ft_train.jsonl")
check_jsonl("policy_ft_val.jsonl")

Checking policy_ft_train.jsonl: 33 lines found.
Checking policy_ft_val.jsonl: 25 lines found.


In [33]:
from openai import OpenAI
client = OpenAI()

def create_finetuning_job():
  # Upload Training File
  training_file = client.files.create(
    file=open("policy_ft_train.jsonl", "rb"),
    purpose="fine-tune"
  )

  # Upload Validation File
  validation_file = client.files.create(
    file=open("policy_ft_val.jsonl", "rb"),
    purpose="fine-tune"
  )

  job = client.fine_tuning.jobs.create(
    training_file=training_file.id,
    validation_file=validation_file.id,
    model="gpt-4o-mini-2024-07-18" 
  )
  print(f"Fine-tuning job started: {job.id}")

  return job.id



In [20]:
import pandas as pd

def ask_and_compare(question, fine_tuned_id):

    # 1. Retrieve Context once (Fairness!)
    # This ensures every model gets the exact same "facts"
    context = get_milvus_context(question, "policy_docs_collection")
    context_str = "\n".join(context)
    
    # 2. Define the models to test
    models = {
        "GPT-4o": "gpt-4o",
        "Fine-Tuned-Model": fine_tuned_id,
        "GPT-4-Turbo": "gpt-4-turbo" # Placeholder for GPT-4.1/5o
    }
    
    comparison_data = []

    question="I need a vacation"
    print(f"\nQuestion: {question}")
    print("-" * 50)

    # 3. Loop through models and get answers
    for name, model_id in models.items():
        print(f"Generating with {name}...")
        try:
            completion = client.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "system", "content": "Answer the question using ONLY the provided context."},
                    {"role": "user", "content": f"Context: {context_str}\n\nQuestion: {question}"}
                ],
                temperature=0 # Keep it deterministic for comparison
            )
            answer = completion.choices[0].message.content
        except Exception as e:
            answer = f"Error: {str(e)}"
        
        comparison_data.append({"Model": name, "Answer": answer})

    # 4. Return as a clean table
    return pd.DataFrame(comparison_data)

# --- EXECUTION ---
# Replace with your actual fine-tuned ID from the OpenAI dashboard
MY_FT_ID = fine_tuned_id 

results_df = ask_and_compare("What is the leave request process?", MY_FT_ID)

# Display results side-by-side
pd.set_option('display.max_colwidth', None)
display(results_df)

Collection Name: policy_docs_collection
Is Empty: False
Entity Count: 11
Load Status: Loaded
Action: Loading collection...

--- Milvus Search Results for: 'What is the leave request process?' ---
 Match 1 | Score: 0.7258 | Content: Employees must submit a leave request for approval....
 Match 2 | Score: 0.3071 | Content: Employees can take an hour break....

Question: I need a vacation
--------------------------------------------------
Generating with GPT-4o...
Generating with Fine-Tuned-Model...
Generating with GPT-4-Turbo...


,Model,Answer
0,GPT-4o,You will need to submit a leave request for approval.
1,Fine-Tuned-Model,"Error: Error code: 404 - {'error': {'message': 'The model `ftjob-2rvwgI8RMhRffbM81LJYo1e0` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}"
2,GPT-4-Turbo,"To take a vacation, you should submit a leave request for approval according to your company's policy. Make sure to check how many days you can take and any specific procedures or forms you need to fill out. Once your leave is approved, you can enjoy your vacation!"


In [37]:
import time
import pandas as pd

# 1. SET YOUR JOB ID
ft_job_id = create_finetuning_job()

def get_finished_model(job_id):
    """Polls OpenAI until the fine-tuned model is ready."""
    while True:
        job = client.fine_tuning.jobs.retrieve(job_id)
        if job.status == "succeeded":
            print(f"Model Ready: {job.fine_tuned_model}")
            return job.fine_tuned_model
        elif job.status == "failed":
            print("Job Failed!")
            return None
        else:
            print(f"⏳ Status: {job.status}... checking in 60s")
            time.sleep(60)

# 2. RUN THE WAIT AND COMPARE
# This is the "Magic Link" between your 5-day journey and the results
ft_model_id = get_finished_model(ft_job_id)

if ft_model_id:
    test_question = "What is the process for requesting leave?"
    
    # Call your existing function (Ensure the cell with this DEF was run!)
    comparison_df = ask_and_compare(test_question, ft_model_id)
    
    print("\n--- THE BATTLE OF THE MODELS ---")
    display(comparison_df)
    
    # Save the result so you never lose it again!
    comparison_df.to_csv("model_comparison_results.csv", index=False)

get_finished_model(ft_job_id)

Fine-tuning job started: ftjob-8yr5Yz0WbLhsAxqtSKdvGGlN
⏳ Status: validating_files... checking in 60s
⏳ Status: validating_files... checking in 60s
⏳ Status: running... checking in 60s
⏳ Status: running... checking in 60s
⏳ Status: running... checking in 60s
⏳ Status: running... checking in 60s
⏳ Status: running... checking in 60s
⏳ Status: running... checking in 60s
⏳ Status: running... checking in 60s
⏳ Status: running... checking in 60s
⏳ Status: running... checking in 60s
⏳ Status: running... checking in 60s
⏳ Status: running... checking in 60s
⏳ Status: running... checking in 60s
⏳ Status: running... checking in 60s
⏳ Status: running... checking in 60s
⏳ Status: running... checking in 60s
⏳ Status: running... checking in 60s
⏳ Status: running... checking in 60s
⏳ Status: running... checking in 60s
⏳ Status: running... checking in 60s
⏳ Status: running... checking in 60s
⏳ Status: running... checking in 60s
⏳ Status: running... checking in 60s
⏳ Status: running... checking in 60s
M

,Model,Answer
0,GPT-4o,You will need to submit a leave request for approval.
1,Fine-Tuned-Model,"Error: Error code: 404 - {'error': {'message': 'The model `ftjob-2rvwgI8RMhRffbM81LJYo1e0` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}"
2,GPT-4-Turbo,"To take a vacation, you should submit a leave request for approval according to your company's policy. Make sure to check how many days you can take and any specific procedures or forms you need to fill out. Once your leave is approved, you can enjoy your vacation!"


Model Ready: ft:gpt-4o-mini-2024-07-18:personal::D6rXgj6D


'ft:gpt-4o-mini-2024-07-18:personal::D6rXgj6D'